# Automate FinOKF question vaults

Submit every subquestion in `questions.json` through the same local endpoints used by the UI. One multi-turn vault is created per question set and provider so the complete answer and evidence graph remain intact.

## Sources and behavior

- Questions: project `questions.json` (with `question.json` accepted as a fallback).
- Credentials: project `.env`; keys are sent with requests and are not written to vaults.
- Ollama: `http://127.0.0.1:11434`.
- Vaults: existing `POST /api/vaults` and `POST /api/chat` UI endpoints.
- Metrics: successful FinOKF and Naive results are upserted into `paper/question_metrics.csv`.

Start the FinOKF UI server before running the execution cell. No financial data is synthesized or replaced.

## Imports and paths

The server module supplies the established model defaults, company resolver, and browser index. Vault mutations still go through the UI API.

In [20]:
import csv
import html
import json
import os
import re
import sys
from pathlib import Path
from urllib import error as urlerror
from urllib import request as urlrequest
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "scripts":
    ROOT = ROOT.parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

import serve_vault as server
from sec2md_processor import load_dotenv

## Configuration

`SKIP_EXISTING=True` avoids repeating expensive model calls when a vault with the expected name is already complete.

In [21]:
BASE_URL = "http://127.0.0.1:8770"
QUESTIONS_PATH = next(
    (path for path in (ROOT / "questions.json", ROOT / "question.json") if path.is_file()),
    ROOT / "questions.json",
)
METRICS_PATH = ROOT / "paper" / "question_metrics.csv"
SKIP_EXISTING = True
INCLUDE_FAILED_METRICS = False

load_dotenv(ROOT / ".env")
PROVIDERS = [
    {"provider": "openai", "model": server.default_llm_model("openai"),
     "api_key": os.environ.get("OPENAI_API_KEY", "")},
    {"provider": "anthropic", "model": server.default_llm_model("anthropic"),
     "api_key": os.environ.get("ANTHROPIC_API_KEY", "")},
    # {"provider": "ollama", "model": server.default_llm_model("ollama"),
    #  "url": "http://127.0.0.1:11434"},
]

## API and input validation

Check the question schema, credentials, UI server, and all provider/model pairs before creating research vaults.

In [22]:
def request_json(method: str, path: str, payload: dict | None = None, timeout: int = 900) -> dict:
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    headers = {"Content-Type": "application/json"} if body is not None else {}
    req = urlrequest.Request(BASE_URL + path, data=body, headers=headers, method=method)
    try:
        with urlrequest.urlopen(req, timeout=timeout) as response:
            result = json.loads(response.read() or b"{}")
    except urlerror.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:1000]
        raise RuntimeError(f"{method} {path} failed ({exc.code}): {detail}") from exc
    if result.get("ok") is False:
        raise RuntimeError(result.get("error") or f"{method} {path} failed")
    return result

question_document = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
question_sets = question_document.get("questions") or []
if not question_sets or any(not item.get("subquestions") for item in question_sets):
    raise ValueError(f"No complete question sets found in {QUESTIONS_PATH}")
for config in PROVIDERS:
    if config.get("provider") in {"openai", "anthropic"} and not config.get("api_key"):
        raise RuntimeError(f"Missing {config['provider']} API key in {ROOT / '.env'}")

request_json("GET", "/api/vaults", timeout=20)
provider_checks = [
    request_json("POST", "/api/verify-provider", {"provider_config": config})
    for config in PROVIDERS
]
provider_checks

[{'ok': True,
  'provider': 'openai',
  'model': 'gpt-5.5',
  'elapsed_ms': 12090.28,
  'model_ms': 12090.248,
  'usage': {'prompt_tokens': 29,
   'completion_tokens': 17,
   'total_tokens': 46,
   'usage_complete': True,
   'ollama_total_ms': 0,
   'ollama_load_ms': 0,
   'request_ids': ['req_8d55e585c977412f912773cc8b5b51d6']}},
 {'ok': True,
  'provider': 'anthropic',
  'model': 'claude-sonnet-5',
  'elapsed_ms': 5553.851,
  'model_ms': 5553.814,
  'usage': {'prompt_tokens': 41,
   'completion_tokens': 4,
   'total_tokens': 45,
   'usage_complete': True,
   'ollama_total_ms': 0,
   'ollama_load_ms': 0,
   'request_ids': ['msg_011CerjpnymChb3A9LJSsi5T']}}]

## Company routing and vault names

The first subquestion determines the starting company node, as it would in the UI. Later comparison questions stay in the same vault, allowing the server to add newly mentioned companies to the cumulative graph.

In [23]:
_browser_index, NODE_BY_ID = server.load_browser_index()

def company_node_for(question_set: dict) -> dict:
    prompt = question_set["subquestions"][0]["prompt"]
    tickers = server.mentioned_company_tickers(prompt, {}, NODE_BY_ID)
    if not tickers:
        raise ValueError(f"Could not resolve a company for {question_set['id']}")
    node = next((item for item in NODE_BY_ID.values()
                 if item.get("type") == "finance.entity" and item.get("ticker") == tickers[0]), None)
    if node is None:
        raise ValueError(f"No finance.entity node found for {tickers[0]}")
    return {key: node.get(key, "") for key in
            ("id", "title", "type", "ticker", "path", "finokf")}

def vault_title(question_set: dict, config: dict) -> str:
    model = re.sub(r"[^A-Za-z0-9._-]+", "-", config["model"]).strip("-")
    return f"{question_set['id']}-{model}"

[(item["id"], company_node_for(item)["ticker"]) for item in question_sets]

[('question-01', 'MSFT'), ('question-02', 'AAPL')]

## Metrics and graph checks

Metrics come from the persisted UI response. Existing CSV rows are preserved and matching rows are atomically updated after every successful turn.

In [24]:
METRIC_FIELDS = ["question_id", "agent", "question", "total_tokens", "elapsed_ms",
                 "provider", "model", "status", "source_vault", "source_turn"]

def compact_question_id(question_id: str) -> str:
    match = re.fullmatch(r"question-(\d+)-(\d+)", question_id)
    return f"{int(match.group(1))}.{int(match.group(2))}" if match else question_id

def metric_rows(question: dict, answers: list[dict], vault: dict, turn: int) -> list[dict]:
    rows = []
    for answer in answers:
        status = "ok" if answer.get("ok", True) else "failed"
        if status == "failed" and not INCLUDE_FAILED_METRICS:
            continue
        metrics = answer.get("metrics") or {}
        rows.append({
            "question_id": compact_question_id(question["id"]),
            "agent": answer.get("agent", ""), "question": question["prompt"],
            "total_tokens": metrics.get("total_tokens", ""),
            "elapsed_ms": metrics.get("total_ms", ""),
            "provider": answer.get("provider", ""), "model": answer.get("model", ""),
            "status": status, "source_vault": vault.get("slug", ""), "source_turn": turn,
        })
    return rows

def upsert_metrics(new_rows: list[dict]) -> None:
    METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    existing = []
    if METRICS_PATH.is_file():
        with METRICS_PATH.open(newline="", encoding="utf-8") as handle:
            existing = list(csv.DictReader(handle))
    keys = ("question_id", "agent", "provider", "model", "source_vault", "source_turn")
    indexed = {tuple(str(row.get(key, "")) for key in keys): row for row in existing}
    for row in new_rows:
        indexed[tuple(str(row.get(key, "")) for key in keys)] = row
    temporary = METRICS_PATH.with_suffix(".csv.tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=METRIC_FIELDS)
        writer.writeheader()
        writer.writerows(indexed.values())
    temporary.replace(METRICS_PATH)

def validate_graph(vault: dict, expected_turns: int) -> None:
    graph_path = ROOT / vault["graph_path"]
    graph = json.loads(graph_path.read_text(encoding="utf-8"))
    if len(vault.get("runs") or []) != expected_turns:
        raise RuntimeError(f"{vault['title']} saved an unexpected number of turns")
    if not isinstance(graph.get("nodes"), list) or not isinstance(graph.get("links"), list):
        raise RuntimeError(f"{graph_path} is not a valid FinOKF graph")
    if expected_turns and not graph["nodes"]:
        raise RuntimeError(f"{graph_path} lost its answer graph")

## Naive and FinOKF comparison

Each completed question displays both full answers followed by the same metadata fields used by the UI's **Compare metadata** panel. Naive is shown first.

In [25]:
def ui_number(value) -> str:
    if value is None:
        return "Unavailable"
    return f"{value:,}" if isinstance(value, (int, float)) else str(value)

def ui_metadata(result: dict) -> dict[str, str]:
    metrics = result.get("metrics") or {}
    usage = metrics.get("usage_complete")
    token_accounting = ("Incomplete: provider usage unavailable for some calls" if usage is False
                        else "Complete" if usage is True else "Not recorded")
    no_inference = result.get("route") == "compiled-program" and metrics.get("total_tokens") == 0
    return {
        "Status": "Answered" if result.get("ok") else "Failed",
        "Provider / model": f"{result.get('provider') or '—'} / {result.get('model') or '—'}",
        "Data access": result.get("data_access") or "—", "Route": result.get("route") or "—",
        "Experiment": result.get("experiment") or "standard",
        "Cache / calculation": result.get("cache_kind") or ("cache hit" if result.get("cache_hit") else "uncached"),
        "Cache hit": "Yes" if result.get("cache_hit") else "No",
        "Elapsed (ms)": ui_number(metrics.get("total_ms")), "Non-model elapsed (ms)": ui_number(metrics.get("local_ms")),
        "Model time (ms)": ui_number(metrics.get("model_ms")), "Model calls": ui_number(metrics.get("model_calls")),
        "Model attempts": ui_number(metrics.get("model_attempts")), "Token accounting": token_accounting,
        "Inference record": "Historical calculation: no model call" if no_inference else "Provider-reported usage",
        "Routing (ms)": ui_number(metrics.get("route_ms")), "Binding / arithmetic (ms)": ui_number(metrics.get("bind_ms")),
        "Vault persistence (ms)": ui_number(metrics.get("persist_ms")), "Input tokens": ui_number(metrics.get("prompt_tokens")),
        "Output tokens": ui_number(metrics.get("completion_tokens")), "Total tokens": ui_number(metrics.get("total_tokens")),
        "Request IDs": ", ".join(metrics.get("request_ids") or []) or "Not recorded",
        "Web searches": ui_number(metrics.get("web_requests")), "Web page requests": ui_number(metrics.get("page_requests")),
        "Web pages read": ui_number(metrics.get("pages_fetched")), "Evidence sources": ui_number(metrics.get("source_count")),
    }


In [26]:
def answers_for_turn(vault: dict, turn: int) -> list[dict]:
    results = [message["result"] for message in vault.get("messages") or []
               if message.get("turn") == turn and isinstance(message.get("result"), dict)]
    return sorted(results, key=lambda item: 0 if item.get("agent") == "naive" else 1)

def display_agent_comparison(question: dict, answers: list[dict]) -> None:
    by_agent = {answer.get("agent"): answer for answer in answers}
    display(Markdown(f"### {question['id']}\n\n> {question['prompt']}"))
    for agent, label, subtitle in (("naive", "Naive", "Independent web research · uncached"),
                                   ("finokf", "FinOKF", "Local evidence + web research · cache enabled")):
        result = by_agent.get(agent) or {"ok": False, "answer": "No result returned."}
        display(Markdown(f"#### {label}\n\n*{subtitle}*\n\n{result.get('answer') or 'No answer returned.'}"))
    naive = ui_metadata(by_agent.get("naive") or {})
    finokf = ui_metadata(by_agent.get("finokf") or {})
    rows = "".join(f"<tr><th style='text-align:left'>{html.escape(label)}</th>"
                   f"<td>{html.escape(naive[label])}</td><td>{html.escape(finokf[label])}</td></tr>"
                   for label in naive)
    display(HTML("<table><caption><strong>Measured for this question</strong></caption>"
                 "<thead><tr><th>Metadata</th><th>Naive</th><th>FinOKF</th></tr></thead>"
                 f"<tbody>{rows}</tbody></table>"))


## Submit one question set

Create the named vault, then send every subquestion as a turn using the same vault ID. The server runs both default agents, writes result files, and rebuilds the graph.

In [27]:
def run_question_set(question_set: dict, config: dict, known_vaults: dict[str, dict]) -> dict:
    title = vault_title(question_set, config)
    expected_turns = len(question_set["subquestions"])
    existing = known_vaults.get(title)
    if SKIP_EXISTING and existing and len(existing.get("runs") or []) == expected_turns:
        validate_graph(existing, expected_turns)
        for turn, question in enumerate(question_set["subquestions"], start=1):
            answers = answers_for_turn(existing, turn)
            display_agent_comparison(question, answers)
            upsert_metrics(metric_rows(question, answers, existing, turn))
        return {"title": title, "status": "skipped-complete", "vault": existing}
    if existing and existing.get("runs"):
        raise RuntimeError(f"{title} exists but is incomplete; resolve it before rerunning.")

    node = company_node_for(question_set)
    vault = request_json("POST", "/api/vaults", {"title": title, "node": node})["vault"]
    for turn, question in enumerate(question_set["subquestions"], start=1):
        result = request_json("POST", "/api/chat", {
            "message": question["prompt"], "stream": False, "provider_config": config,
            "vault_id": vault["vault_id"], "markdown": "", "node": node, "neighbors": [],
        })
        if result.get("persistence_error"):
            raise RuntimeError(result["persistence_error"])
        vault = result["vault"]
        validate_graph(vault, turn)
        answers = sorted(result.get("answers") or [], key=lambda item: 0 if item.get("agent") == "naive" else 1)
        display_agent_comparison(question, answers)
        upsert_metrics(metric_rows(question, answers, vault, turn))
        print(f"{title}: completed {question['id']} ({turn}/{expected_turns})")
    return {"title": title, "status": "completed", "vault": vault}


## Execution plan

Review the names before starting. With the current input this creates six vaults: question sets 01 and 02 for the OpenAI, Anthropic, and Ollama default models.

In [28]:
execution_plan = [
    {"question_set": item["id"], "provider": config["provider"],
     "model": config["model"], "vault": vault_title(item, config),
     "turns": len(item["subquestions"])}
    for item in question_sets for config in PROVIDERS
]
execution_plan

[{'question_set': 'question-01',
  'provider': 'openai',
  'model': 'gpt-5.5',
  'vault': 'question-01-gpt-5.5',
  'turns': 3},
 {'question_set': 'question-01',
  'provider': 'anthropic',
  'model': 'claude-sonnet-5',
  'vault': 'question-01-claude-sonnet-5',
  'turns': 3},
 {'question_set': 'question-02',
  'provider': 'openai',
  'model': 'gpt-5.5',
  'vault': 'question-02-gpt-5.5',
  'turns': 4},
 {'question_set': 'question-02',
  'provider': 'anthropic',
  'model': 'claude-sonnet-5',
  'vault': 'question-02-claude-sonnet-5',
  'turns': 4}]

## Run all providers

This is the long-running cell. Providers run sequentially so activity stays attributable; progress prints after every persisted turn.

In [29]:
known_vaults = {item.get("title"): item for item in request_json("GET", "/api/vaults")["vaults"]}
run_summary = []
for question_set in question_sets:
    for config in PROVIDERS:
        outcome = run_question_set(question_set, config, known_vaults)
        vault = outcome["vault"]
        run_summary.append({
            "vault": outcome["title"], "provider": config["provider"],
            "model": config["model"], "status": outcome["status"],
            "turns": len(vault.get("runs") or []), "graph_path": vault.get("graph_path"),
        })
        known_vaults[outcome["title"]] = vault

### question-01-01

> How did the profitability of Microsoft's growth change across FY2023, FY2024 and FY2025? Assess consolidated operating leverage and whether the profitability of additional revenue strengthened or weakened.

#### Naive

*Independent web research · uncached*

On reported GAAP consolidated results, Microsoft’s average profitability improved each year, but the **profitability of incremental growth peaked in FY2024 and then weakened in FY2025**.

| Fiscal year | Revenue ($m) | Operating income ($m) | Operating margin | Δ Revenue YoY ($m) | Δ Op. income YoY ($m) | Incremental operating margin | Operating leverage* |
|---|---:|---:|---:|---:|---:|---:|---:|
| FY2023 | 211,915 | 88,523 | 41.8% | 13,645 | 5,140 | 37.7% | 0.90x |
| FY2024 | 245,122 | 109,433 | 44.6% | 33,207 | 20,910 | 63.0% | 1.51x |
| FY2025 | 281,724 | 128,528 | 45.6% | 36,602 | 19,095 | 52.2% | 1.17x |

\*Operating leverage = operating-income growth rate ÷ revenue growth rate.

**Assessment.** FY2023 showed weak/negative consolidated operating leverage: revenue rose 6.9%, while operating income rose only 6.2%, so operating income grew slightly slower than revenue. Incremental operating margin was 37.7%, below Microsoft’s 41.8% FY2023 operating margin.

FY2024 was the inflection: revenue grew 15.7%, operating income grew 23.6%, and incremental operating margin jumped to 63.0%. That means the FY2024 revenue growth was much more profitable than the existing revenue base, producing strong operating leverage.

FY2025 still had positive operating leverage—operating income grew 17.4% versus revenue growth of 14.9%—but the incremental margin fell to 52.2%. So the profitability of additional revenue **weakened versus FY2024**, even though it remained above Microsoft’s average FY2025 operating margin and therefore still expanded consolidated margin.

Key caveats: FY2023 reported operating income was affected by two disclosed items: a useful-life accounting estimate change increased FY2023 operating income by $3.7 billion, while a Q2 charge reduced it by $1.2 billion. Also, Microsoft says FY2025 segment recasts did **not** affect consolidated income statements. Microsoft’s FY2025 filing attributes gross-margin pressure partly to scaling AI infrastructure, consistent with lower incremental profitability in FY2025 despite continued margin expansion.

**Sources:** Microsoft FY2023 Annual Report: https://www.microsoft.com/investor/reports/ar23/index.html ; Microsoft FY2025 Annual Report: https://www.microsoft.com/investor/reports/ar25/index.html

#### FinOKF

*Local evidence + web research · cache enabled*

**Conclusion:** Microsoft’s consolidated profitability of growth **strengthened sharply in FY2024, then still strengthened but less in FY2025**. Operating leverage was positive in both growth intervals because operating income grew faster than revenue, but the profitability of each additional revenue dollar **weakened from FY2024 to FY2025**.

| Fiscal year | Revenue (USD mm) | Operating income (USD mm) | Operating margin | Change vs prior year | Incremental operating margin |
|---|---:|---:|---:|---:|---:|
| FY2023 | 211,915 | 88,523 | 41.77% | — | — |
| FY2024 | 245,122 | 109,433 | 44.64% | +287.1 bps | 62.97% |
| FY2025 | 281,724 | 128,528 | 45.62% | +97.8 bps | 52.17% |

**Assessment:**  
From FY2023 to FY2024, revenue rose **$33.207 billion** while operating income rose **$20.910 billion**, producing a **62.97% incremental operating margin**, well above the FY2023 base margin of **41.77%**. That indicates strong consolidated operating leverage.

From FY2024 to FY2025, revenue rose a larger **$36.602 billion**, but operating income rose **$19.095 billion**, implying a lower **52.17% incremental operating margin**. This was still above the FY2024 margin of **44.64%**, so operating leverage remained positive, but the incremental profitability of growth **moderated**.

These reported-period ratios show how profit scaled with revenue; they **do not establish causal drivers or future persistence**.

**Source:** `filings/MSFT/MSFT-FY2025-10-K-2025-07-30-0000950170-25-100235.md`

Metadata,Naive,FinOKF
Status,Answered,Answered
Provider / model,openai / gpt-5.5,openai / gpt-5.5
Data access,Public web search and page retrieval; independent numerical review; uncached,Local filings and stock prices from data/prices; web research for missing non-price evidence
Route,web-research,openai-measured-synthesis
Experiment,standard baseline,standard
Cache / calculation,uncached,calculation-lru
Cache hit,No,Yes
Elapsed (ms),"142,266.324","13,126.153"
Non-model elapsed (ms),Unavailable,80.649
Model time (ms),"74,773.37","13,045.504"


question-01-gpt-5.5: completed question-01-01 (1/3)


...

### question-02-04

> How did MSFT's consolidated operating leverage change between FY2024 and FY2025 as compared to Apple?

#### Naive

*Independent web research · uncached*

Naive could not complete this answer: Anthropic returned no text output.

#### FinOKF

*Local evidence + web research · cache enabled*

**Conclusion: MSFT showed stronger consolidated operating leverage than Apple in FY2025, with both margin expansion and incremental profitability well above Apple's.**

MSFT's operating margin rose 97.8 bps (44.64%→45.62%) versus Apple's 46.1 bps (31.51%→31.97%) — roughly 2x the improvement. More notably, MSFT's incremental operating margin (the margin on new revenue) was 52.17%, exceeding its base FY2024 margin of 44.64%, indicating each new revenue dollar was more profitable than the existing base. Apple's incremental margin of 39.14% was *below* its FY2024 base margin of 31.51%... actually 39.14% is above 31.51%, so Apple's new revenue was also more profitable than its base, but the effect was smaller in both absolute margin gain and incremental spread.

MSFT's revenue grew faster in dollar terms relative to its base ($36.6B on $245B) than Apple's ($25.1B on $391B), and converted that growth into disproportionately higher operating income growth.

| Metric | MSFT | AAPL |
|---|---:|---:|
| FY24 margin | 44.64% | 31.51% |
| FY25 margin | 45.62% | 31.97% |
| Δ margin (bps) | +97.8 | +46.1 |
| Incremental margin | 52.17% | 39.14% |

**What this shows:** consolidated operating leverage improved for both, more so for MSFT.
**What this cannot establish:** segment mix, cost drivers (e.g., AI capex, R&D allocation), or whether the pattern persists into FY2026 — ratios alone don't reveal causation.

**Sources:**
- `filings/MSFT/MSFT-FY2025-10-K-2025-07-30-0000950170-25-100235.md`
- `filings/AAPL/AAPL-FY2025-10-K-2025-10-31-0000320193-25-000079.md`

Metadata,Naive,FinOKF
Status,Failed,Answered
Provider / model,anthropic / claude-sonnet-5,anthropic / claude-sonnet-5
Data access,Public web search and page retrieval; independent numerical review; uncached,Local filings and stock prices from data/prices; web research for missing non-price evidence
Route,web-research,anthropic-measured-synthesis
Experiment,standard baseline,standard
Cache / calculation,uncached,calculation-lru
Cache hit,No,Yes
Elapsed (ms),"119,190.68","11,824.029"
Non-model elapsed (ms),Unavailable,92.816
Model time (ms),"25,357.007","11,731.213"


question-02-claude-sonnet-5: completed question-02-04 (4/4)


## Results

Confirm every vault has its expected turns and graph path, then refresh Home in the FinOKF UI.

In [30]:
run_summary

[{'vault': 'question-01-gpt-5.5',
  'provider': 'openai',
  'model': 'gpt-5.5',
  'status': 'completed',
  'turns': 3,
  'graph_path': 'data/vaults/answers/msft-20260908t221140794429z-fdd1407f2e64/graph.json'},
 {'vault': 'question-01-claude-sonnet-5',
  'provider': 'anthropic',
  'model': 'claude-sonnet-5',
  'status': 'completed',
  'turns': 3,
  'graph_path': 'data/vaults/answers/msft-20260908t221821835917z-5349c4edd9c6/graph.json'},
 {'vault': 'question-02-gpt-5.5',
  'provider': 'openai',
  'model': 'gpt-5.5',
  'status': 'completed',
  'turns': 4,
  'graph_path': 'data/vaults/answers/aapl-20260908t222452292244z-bf2f6699dea0/graph.json'},
 {'vault': 'question-02-claude-sonnet-5',
  'provider': 'anthropic',
  'model': 'claude-sonnet-5',
  'status': 'completed',
  'turns': 4,
  'graph_path': 'data/vaults/answers/aapl-20260908t223726364264z-aa4c6319a7f0/graph.json'}]